# Veil In-House Image Detector — Kaggle Training Kernel

Runs the end-to-end pipeline (`detector-trainer/run_pipeline.py`) on Kaggle GPU:
**manifest -> train_resnet -> train_clip -> evaluate**, writing all outputs to
`/kaggle/working` for retrieval via `kaggle kernels output`.

## How the code gets onto Kaggle
This kernel needs the `detector-trainer/` source tree importable. Two supported ways:

1. **Git clone (default below, needs `enable_internet: true`)** — clone the repo
   at run time. Simplest; the `CODE` cell does this.
2. **Attached utility dataset** — push `detector-trainer/` as a Kaggle Dataset
   (`kaggle datasets create/version`), attach it, and point `CODE_DIR` at
   `/kaggle/input/<your-code-dataset>/detector-trainer`. Then set `USE_GIT = False`.

See `detector-trainer/kernels/README.md` for the exact CLI loop.

## 0. Config
Edit `REPO_URL` / `BRANCH` (git path) or `CODE_DIR` (attached-dataset path).

In [ ]:
import os, subprocess, sys, glob, json

USE_GIT   = True   # False -> use an attached code dataset instead of cloning
REPO_URL  = 'https://github.com/<YOUR_GH_USER>/ai-detector-repo.git'  # fill in
BRANCH    = 'detector-model'
CODE_DIR  = '/kaggle/working/ai-detector-repo/detector-trainer'  # git target
# If USE_GIT is False, set CODE_DIR to the attached dataset path, e.g.:
# CODE_DIR = '/kaggle/input/veil-detector-code/detector-trainer'

OUT_DIR = '/kaggle/working/veil_run'
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Get the code

In [ ]:
if USE_GIT:
    if not os.path.isdir('/kaggle/working/ai-detector-repo'):
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL,
             '/kaggle/working/ai-detector-repo'], check=True)
assert os.path.isdir(CODE_DIR), f'code dir not found: {CODE_DIR}'
print('code dir:', CODE_DIR)
print(sorted(os.listdir(CODE_DIR)))

## 2. Install dependencies
Kaggle has torch preinstalled; this pulls open_clip / imagehash / matching pins.

In [ ]:
req = os.path.join(CODE_DIR, 'requirements.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', req], check=True)
print('deps installed')

## 3. Locate the attached datasets
Kaggle mounts `dataset_sources` (see `kernel-metadata.json`) read-only under
`/kaggle/input/<dataset-slug>/`. We pass BOTH roots to `--data-root`; the
pipeline auto-discovers per-generator / category subfolders and classifies them
(real vs each generator). Wild generators (midjourney/dalle3/flux) are routed to
`test_wild` by the split logic. Drop any self-generated dalle3/flux images into a
folder named `dalle3`/`flux` under a data root and they are picked up automatically.

In [ ]:
DATA_ROOTS = [
    '/kaggle/input/unbiased-tiny-genimage',
    '/kaggle/input/ai-vs-real-images-dataset',
]
for r in DATA_ROOTS:
    print(r, '->', os.path.isdir(r))
    if os.path.isdir(r):
        for sub in sorted(os.listdir(r))[:12]:
            print('   ', sub)

## 4. Run the full pipeline
All four stages to `/kaggle/working/veil_run`. Adjust epochs for the GPU budget.

In [ ]:
cmd = [
    sys.executable, 'run_pipeline.py',
    '--data-root', *DATA_ROOTS,
    '--manifest', os.path.join(OUT_DIR, 'manifest.csv'),
    '--out', OUT_DIR,
    '--stages', 'all',
    '--pretrained',              # ImageNet init for the ResNet baseline
    '--resnet-epochs', '10',
    '--clip-backbone', 'ViT-L-14',
    '--clip-epochs', '200',
    '--num-workers', '2',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=CODE_DIR, check=True)

## 5. Inspect outputs
`report.md`, plots, per-model predictions, checkpoints and `results.json` all land
under `/kaggle/working/veil_run`. Pull them back locally with:
```
kaggle kernels output <KAGGLE_USERNAME>/veil-detector-train -p ./kaggle_out
```

In [ ]:
for p in sorted(glob.glob(os.path.join(OUT_DIR, '**', '*'), recursive=True)):
    if os.path.isfile(p):
        print(p)
print('\n----- report.md -----')
rp = os.path.join(OUT_DIR, 'report', 'report.md')
if os.path.exists(rp):
    print(open(rp).read())